# Forecasting GDP Growth
This project tasks you with forecasting quarterly nominal GDP growth rate in the United States. The dataset contains estimated nominal gross domestic product (GDP) annual growth rate in each quarter from 1990 to 2020, released by the Bureau of Economic Analysis (BEA) of the United States Department of Commerce. The estimate is the “third” or "final" estimate of GDP issued in the third month after the relevant quarter which is based on more complete source data than were available for the "second" estimate issued in the second month and the "advance" estimate issued in the first month after the relevant quarter. You may refer to the [BEA website](https://www.bea.gov/resources/learning-center/what-to-know-gdp) for more information on the estimate of GDP in the United States. You will train models to forecast the BEA's final GDP estimate for a quarter. Specifically you are required to answer this question: Is accounting information useful for GDP forecast? You will be allowed to use any outside data to build your model.

The evaluation pages describes how submissions will be scored and how students should format their submissions. --- The evaluation metric for this competition is [RMSE](https://en.wikipedia.org/wiki/Root-mean-square_deviation). RMSE is a measure of the accuracy of your predictions.

**Files**
- GDP_Forecast_train.csv - the training set
- GDP_Forecast_test.csv - the test set
- GDP_Forecast_sampleSubmission.csv - a sample submission file in the correct format

**Columns**
- YQ - The year and quarter variable. 1990Q1 refers to the first calendar quarter of 1990, ie, Jan - March 1990
- NGDP - nominal GDP growth rate in the quarter of the year. It is obtained from the BEA final estimate which is available at the BEA website.

**Features** (no features data provided)

Following Datar et al. (2020), you may consider the following data as your predictors/features:
- Accounting data - through COMPUSTAT on WRDS
- Stock price/return data - through CRSP on WRDS
- Analysts' forecast of GDP - through Survey of Professional Forecasters
- Any other data

In [94]:
import pandas as pd

train = pd.read_csv("data/GDP_Forecast_train.csv")
test = pd.read_csv("data/GDP_Forecast_test.csv")

In [95]:
def create_submission(test_df, y_pred, filename):
    df = pd.DataFrame({"YQ": test_df["YQ"], "NGDP": y_pred})
    df.to_csv(f"attempts/{filename}.csv", index = False)

In [96]:
# splitting year-quarter to check for traits in yearly patterns vs quarterly pattern

# ensure year is an integer
train["year"] = train["YQ"].str[:4].astype(int)
test["year"] = test["YQ"].str[:4].astype(int)
# keep qtr as string / category
train["qtr"] = train["YQ"].str[4:]
test["qtr"] = test["YQ"].str[4:]
train.head()

,YQ,NGDP,year,qtr
0,1990Q1,9.0,1990,Q1
1,1990Q2,6.1,1990,Q2
2,1990Q3,3.7,1990,Q3
3,1990Q4,-0.7,1990,Q4
4,1991Q1,2.0,1991,Q1


### Base model for reference

In [97]:
import statsmodels.formula.api as smf

# running base linear model
base_model = smf.ols("NGDP ~ year + qtr", train).fit()
# printing summary of base_model
print(base_model.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.118
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     3.324
Date:                Tue, 03 Mar 2026   Prob (F-statistic):             0.0134
Time:                        15:36:22   Log-Likelihood:                -244.91
No. Observations:                 104   AIC:                             499.8
Df Residuals:                      99   BIC:                             513.0
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    219.8404     68.424      3.213      0.0

In [98]:
import numpy as np
from sklearn.metrics import mean_squared_error

# get predicted values based off model
y_pred = base_model.predict(test)
y_true = test["NGDP"]
create_submission(test, y_pred, "0_base_forecast")
# evaluate using RMSE:
print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred))}")

RMSE: 2.96495609125275


Interpretation: To be discussed

### How Accounting Data affects GDP (COMPUSTAT on WRDS)
- [Compustat North America](https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/north-america-daily/) is a database of U.S. and Canadian fundamental and market information on active and inactive publicly held companies. It provides more than 300 annual and 100 quarterly Income Statement, Balance Sheet, Statement of Cash Flows, and supplemental data items. For most companies, annual history is available back to 1950 and quarterly history back to 1962 with monthly market history back to 1962.
- To consider [Execucomp](https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/execucomp/): Executive compensation data collected directly from each company’s annual proxy (DEF14A SEC form)



In [ ]:
# Using Compustat North America: Fundamentals Quarterly
# https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/north-america-daily/fundamentals-quarterly/
# NOTE: currency in USD, data in STD format

compustat = pd.read_csv("data/compustat_quarterly_financials.csv")
## dropping columns with no unique data
compustat.drop(columns = ["curcdq", "datafmt", "indfmt", "consol"], inplace = True)
## dropping columns without too many missing data
# print(compustat[compustat["ltq"].notna()].shape) # 26666 out of 85349
compustat.drop(columns = ["ltq"], inplace = True)
## droping costat - only 2 options
compustat.drop(columns = ["costat"], inplace = True)
compustat.head()

,gvkey,datadate,conm,datacqtr,atq,niq
0,1178,1990-03-31,AFFILIATED BANKSHARES COLO,1990Q1,2597.647,2.445
1,1178,1990-06-30,AFFILIATED BANKSHARES COLO,1990Q2,2645.271,3.234
2,1178,1990-09-30,AFFILIATED BANKSHARES COLO,1990Q3,2682.906,3.253
3,1178,1990-12-31,AFFILIATED BANKSHARES COLO,1990Q4,2738.097,0.897
4,1178,1991-03-31,AFFILIATED BANKSHARES COLO,1991Q1,2655.894,3.536


** NOTE TO DISCUSS, REMOVE COSTAT OR NO

In [100]:
print("total values present:")
print(f"total shape: {compustat.shape}")
print(f"datadate: {compustat[compustat["datadate"].notna()].shape[0]}\
        conm: {compustat[compustat["conm"].notna()].shape[0]}\
        datacqtr: {compustat[compustat["datacqtr"].notna()].shape[0]}")
print(f"atq: {compustat[compustat["atq"].notna()].shape[0]}\
        niq: {compustat[compustat["niq"].notna()].shape[0]}")

total values present:
total shape: (98258, 6)
datadate: 98258        conm: 98258        datacqtr: 98258
atq: 83551        niq: 85553


- (gvkey) Global Company Key
- (conm) Company Name
- (datacqtr) Calendar Data Year and Quarter
- (atq) Assets - Total
- (ltq) Liabilities - Total
- (niq) Net Income (Loss)

In [101]:
train_comp_macro = train.merge(compustat, how = "left", left_on = "YQ", right_on = "datacqtr")
test_comp_macro = test.merge(compustat, how = "left", left_on = "YQ", right_on = "datacqtr")
train_comp_macro.head()

,YQ,NGDP,year,qtr,gvkey,datadate,conm,datacqtr,atq,niq
0,1990Q1,9.0,1990,Q1,1178,1990-03-31,AFFILIATED BANKSHARES COLO,1990Q1,2597.647,2.445
1,1990Q1,9.0,1990,Q1,1194,1990-03-31,AHMANSON (H F) & CO,1990Q1,45984.500,65.978
2,1990Q1,9.0,1990,Q1,1416,1990-03-31,AMERICAN CAPITAL CORP,1990Q1,5257.766,NaN
3,1990Q1,9.0,1990,Q1,1553,1990-03-31,AMERICAN SVGS/FL FSB,1990Q1,NaN,0.698
4,1990Q1,9.0,1990,Q1,3431,1990-03-31,CONSTELLATION BANCORP,1990Q1,NaN,NaN


In [102]:
# creating model with accounting macro data
# no need to model gvkey and conm on the same model
comp_macro_model = smf.ols("NGDP ~ year + qtr + conm + atq + niq", train_comp_macro).fit()
# printing summary of comp_macro_model
print(comp_macro_model.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.156
Model:                            OLS   Adj. R-squared:                  0.130
Method:                 Least Squares   F-statistic:                     6.103
Date:                Tue, 03 Mar 2026   Prob (F-statistic):               0.00
Time:                        15:36:52   Log-Likelihood:            -1.6624e+05
No. Observations:               71262   AIC:                         3.367e+05
Df Residuals:                   69166   BIC:                         3.559e+05
Df Model:                        2095                                         
Covariance Type:            nonrobust                                         
                                           coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------


Insights: Takes very long, and not what we want

In [103]:
compustat_macro = compustat.groupby("datacqtr").agg({"atq": "sum", "niq": "sum"}).reset_index()
# convert net income to percentage because GDP is growth rate
compustat_macro["atq_growth"] = compustat_macro["atq"].pct_change() * 100
compustat_macro["niq_growth"] = compustat_macro["niq"].pct_change() * 100
# converting infinite values to NaN
compustat_macro = compustat_macro.replace([np.inf, -np.inf], np.nan)
# creating a forensic lag
compustat_macro["niq_growth_lag1"] = compustat_macro["niq_growth"].shift(1)
compustat_macro["atq_growth_lag1"] = compustat_macro["atq_growth"].shift(1)
compustat_macro.head()

,datacqtr,atq,niq,atq_growth,niq_growth,niq_growth_lag1,atq_growth_lag1
0,1989Q4,0.000,0.000,NaN,NaN,NaN,NaN
1,1990Q1,3187273.043,4684.085,NaN,NaN,NaN,NaN
2,1990Q2,3412321.689,6396.105,7.060852,36.549721,NaN,NaN
3,1990Q3,3473969.743,3093.203,1.806631,-51.639271,36.549721,7.060852
4,1990Q4,3426461.019,-49.699,-1.367563,-101.606716,-51.639271,1.806631


In [104]:
# merging updated macro to train and test sets
train_comp_macro = train.merge(compustat_macro, left_on = "YQ", right_on = "datacqtr", how = "left")
test_comp_macro = test.merge(compustat_macro, left_on = "YQ", right_on = "datacqtr", how = "left")

In [105]:
# creating model with accounting cleaned macro data
comp_macro_model2 = smf.ols("NGDP ~ year + qtr + atq + niq \
                            + atq_growth + niq_growth \
                            + niq_growth_lag1 + atq_growth_lag1", train_comp_macro).fit()
# printing summary of comp_macro_model2
print(comp_macro_model2.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.396
Model:                            OLS   Adj. R-squared:                  0.330
Method:                 Least Squares   F-statistic:                     5.967
Date:                Tue, 03 Mar 2026   Prob (F-statistic):           6.76e-07
Time:                        15:36:53   Log-Likelihood:                -220.42
No. Observations:                 102   AIC:                             462.8
Df Residuals:                      91   BIC:                             491.7
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept         235.0200    256.495     

In [106]:
# get predicted values based off model
y_pred2 = comp_macro_model2.predict(test_comp_macro)
y_true2 = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred2, "1_acct_macro1_forecast")

# evaluate using RMSE:
print(f"RMSE Base: {np.sqrt(mean_squared_error(y_true, y_pred))}")
print(f"RMSE w/ Accounting Macro: {np.sqrt(mean_squared_error(y_true2, y_pred2))}")

RMSE Base: 2.96495609125275
RMSE w/ Accounting Macro: 4.200668811019781


**Stage 1**: removing atq (p-value: 0.281), niq_growth (p-value: 0.312) and niq_growth_lag1 (p-value: 0.681)

In [107]:
# creating model with accounting cleaned macro data
comp_macro_model3 = smf.ols("NGDP ~ year + qtr + niq + atq_growth + atq_growth_lag1", train_comp_macro).fit()
# printing summary of comp_macro_model3
print(comp_macro_model3.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.378
Model:                            OLS   Adj. R-squared:                  0.332
Method:                 Least Squares   F-statistic:                     8.166
Date:                Tue, 03 Mar 2026   Prob (F-statistic):           9.62e-08
Time:                        15:36:53   Log-Likelihood:                -221.91
No. Observations:                 102   AIC:                             459.8
Df Residuals:                      94   BIC:                             480.8
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept         488.4112     75.121     

In [108]:
# get predicted values based off model
y_pred3 = comp_macro_model3.predict(test_comp_macro)
y_true3 = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred3, "2_acct_macro2_forecast")

# evaluate using RMSE:
print(f"RMSE Base: {np.sqrt(mean_squared_error(y_true, y_pred))}")
print(f"RMSE w/ Accounting Macro: {np.sqrt(mean_squared_error(y_true2, y_pred2))}")
print(f"RMSE w/ Accounting Macro V2: {np.sqrt(mean_squared_error(y_true3, y_pred3))}")

RMSE Base: 2.96495609125275
RMSE w/ Accounting Macro: 4.200668811019781
RMSE w/ Accounting Macro V2: 3.8884866286383355


Insights: improved RMSE, but not as good as Base

**Stage 2**: removing atq_growth_lag1 (p-value: 0.115)

In [109]:
# creating model with accounting cleaned macro data
comp_macro_model4 = smf.ols("NGDP ~ year + qtr + niq + atq_growth", train_comp_macro).fit()
# printing summary of comp_macro_model4
print(comp_macro_model4.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.363
Model:                            OLS   Adj. R-squared:                  0.324
Method:                 Least Squares   F-statistic:                     9.133
Date:                Tue, 03 Mar 2026   Prob (F-statistic):           6.69e-08
Time:                        15:36:53   Log-Likelihood:                -224.96
No. Observations:                 103   AIC:                             463.9
Df Residuals:                      96   BIC:                             482.4
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    482.9765     74.723      6.464      0.0

In [110]:
# get predicted values based off model
y_pred4 = comp_macro_model4.predict(test_comp_macro)
y_true4 = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred4, "3_acct_macro3_forecast")

# evaluate using RMSE:
print(f"RMSE Base: {np.sqrt(mean_squared_error(y_true, y_pred))}")
print(f"RMSE w/ Accounting Macro: {np.sqrt(mean_squared_error(y_true2, y_pred2))}")
print(f"RMSE w/ Accounting Macro V2: {np.sqrt(mean_squared_error(y_true3, y_pred3))}")
print(f"RMSE w/ Accounting Macro V3: {np.sqrt(mean_squared_error(y_true4, y_pred4))}")

RMSE Base: 2.96495609125275
RMSE w/ Accounting Macro: 4.200668811019781
RMSE w/ Accounting Macro V2: 3.8884866286383355
RMSE w/ Accounting Macro V3: 3.638232349956233


Insights: improved RMSE, but still not as good as Base

**Stage 3**: removing atq_growth (p-value: 0.147)

In [111]:
# creating model with accounting cleaned macro data
comp_macro_model5 = smf.ols("NGDP ~ year + qtr + niq", train_comp_macro).fit()
# printing summary of comp_macro_model5
print(comp_macro_model5.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.345
Model:                            OLS   Adj. R-squared:                  0.312
Method:                 Least Squares   F-statistic:                     10.33
Date:                Tue, 03 Mar 2026   Prob (F-statistic):           5.63e-08
Time:                        15:36:53   Log-Likelihood:                -229.45
No. Observations:                 104   AIC:                             470.9
Df Residuals:                      98   BIC:                             486.8
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    485.4484     74.782      6.492      0.0

In [112]:
# get predicted values based off model
y_pred5 = comp_macro_model5.predict(test_comp_macro)
y_true5 = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred5, "4_acct_macro4_forecast")

# evaluate using RMSE:
print(f"RMSE Base: {np.sqrt(mean_squared_error(y_true, y_pred))}")
print(f"RMSE w/ Accounting Macro: {np.sqrt(mean_squared_error(y_true2, y_pred2))}")
print(f"RMSE w/ Accounting Macro V2: {np.sqrt(mean_squared_error(y_true3, y_pred3))}")
print(f"RMSE w/ Accounting Macro V3: {np.sqrt(mean_squared_error(y_true4, y_pred4))}")
print(f"RMSE w/ Accounting Macro V4: {np.sqrt(mean_squared_error(y_true5, y_pred5))}")

RMSE Base: 2.96495609125275
RMSE w/ Accounting Macro: 4.200668811019781
RMSE w/ Accounting Macro V2: 3.8884866286383355
RMSE w/ Accounting Macro V3: 3.638232349956233
RMSE w/ Accounting Macro V4: 3.3948414864271554


In [113]:
# creating model with accounting cleaned macro data
comp_macro_model6 = smf.ols("NGDP ~ year + niq", train_comp_macro).fit()
# printing summary of comp_macro_model6
print(comp_macro_model6.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.326
Model:                            OLS   Adj. R-squared:                  0.312
Method:                 Least Squares   F-statistic:                     24.40
Date:                Tue, 03 Mar 2026   Prob (F-statistic):           2.26e-09
Time:                        15:36:53   Log-Likelihood:                -230.97
No. Observations:                 104   AIC:                             467.9
Df Residuals:                     101   BIC:                             475.9
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    484.9785     74.022      6.552      0.0

In [114]:
# get predicted values based off model
y_pred6 = comp_macro_model6.predict(test_comp_macro)
y_true6 = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred6, "5_acct_macro5_forecast")

# evaluate using RMSE:
print(f"RMSE Base: {np.sqrt(mean_squared_error(y_true, y_pred))}")
print(f"RMSE w/ Accounting Macro: {np.sqrt(mean_squared_error(y_true2, y_pred2))}")
print(f"RMSE w/ Accounting Macro V2: {np.sqrt(mean_squared_error(y_true3, y_pred3))}")
print(f"RMSE w/ Accounting Macro V3: {np.sqrt(mean_squared_error(y_true4, y_pred4))}")
print(f"RMSE w/ Accounting Macro V4: {np.sqrt(mean_squared_error(y_true5, y_pred5))}")
print(f"RMSE w/ Accounting Macro without Quarter: {np.sqrt(mean_squared_error(y_true6, y_pred6))}")

RMSE Base: 2.96495609125275
RMSE w/ Accounting Macro: 4.200668811019781
RMSE w/ Accounting Macro V2: 3.8884866286383355
RMSE w/ Accounting Macro V3: 3.638232349956233
RMSE w/ Accounting Macro V4: 3.3948414864271554
RMSE w/ Accounting Macro without Quarter: 3.3808674839015227


Insights: after feeding output to Kaggle, it was recognised that
`atq + niq + atq_growth + niq_growth + niq_growth_lag1 + atq_growth_lag1`
performed the best even though its RMSE was highest

### How Stock Price / Return Data affects GDP (CRSP on WRDS)
1. Essential Variables (The "Core" Signal)
- (dlycaldt) Daily Calendar Date: daily moves into Year-Quarter (YQ) buckets
- (dlytotret) Daily Total Return on Index: This includes dividends. In forensic finance, this is the most powerful leading indicator. It represents the market's "real-time" discount of future economic growth.
- (dlytotval) Index Total Capitalization Value: This is the aggregate dollar value of all firms in the index. Use this to measure the Wealth Effect. If this value rises significantly, consumer spending (the biggest part of GDP) usually follows 1–2 quarters later.

2. Forensic Variables (To Lower Your RMSE)
- (dlytotind) Daily Index Level: Useful if you want to calculate the "momentum" of the market (e.g., comparing the current level to a 200-day moving average).
- (dlyincret) Daily Income Return on Index: This isolates the dividend component. A high income return relative to price return often signals a "flight to quality," which forensically precedes economic slowdowns.
- (dlyusdcnt) Used Count: The number of firms currently in the index. Rapid changes in the count of firms can signal structural shifts in the economy (like a wave of IPOs or bankruptcies).

In [115]:
# Using CRSP Daily Index and Portfolios on S&P 500
# https://wrds-www.wharton.upenn.edu/pages/get-data/center-research-security-prices-crsp/annual-update/index-version-2/daily-index-and-portfolios-on-sp-500/

crsp_daily_macro = pd.read_csv("data/crsp_snp500_indexes.csv")
## dropping columns with no unique data
# print(crsp_daily_macro[crsp_daily_macro["INDNO"] != 1000500].shape)
crsp_daily_macro.drop(columns = "INDNO", inplace = True)

## creating year, month and quarter identifiers
crsp_daily_macro["DlyCalDt"] = pd.to_datetime(crsp_daily_macro["DlyCalDt"])
crsp_daily_macro.set_index("DlyCalDt", inplace = True)
crsp_daily_macro

,DlyTotRet,DlyTotInd,DlyIncRet,DlyUsdCnt,DlyTotVal
DlyCalDt,,,,,
1990-01-02,0.017527,633.75,0.000000,500,2371902979
1990-01-03,-0.002028,632.47,0.000020,500,2367046297
1990-01-04,-0.007913,627.46,0.000434,500,2347288970
1990-01-05,-0.010069,621.14,0.000023,500,2323602293
1990-01-08,0.004798,624.12,0.000088,500,2334547314
...,...,...,...,...,...
2020-12-24,0.003623,12743.81,0.000103,505,32666322924
2020-12-28,0.008919,12857.47,0.000000,505,32957660480
2020-12-29,-0.002235,12828.73,0.000000,505,32884007696


In [116]:
## create quarterly data
crsp_qtr_macro = crsp_daily_macro.resample("Q").agg({
    "DlyTotRet": [
        ("qtr_ret", lambda x: (1+x).prod() - 1),    # compounded total return
        ("qtr_vola", "std")                         # quarterly volatility
    ],
    "DlyTotInd": [("qtr_index_level", "last")],     # final index level
    "DlyIncRet": [("qtr_income_ret", "sum")],       # total income return on index
    "DlyUsdCnt": [("avg_firm_count", "mean")]       # average number of firms
})

crsp_qtr_macro.columns = crsp_qtr_macro.columns.droplevel(0)
crsp_qtr_macro.reset_index(inplace = True)
## creating YQ column and moving it to first column
crsp_qtr_macro["YQ"] = crsp_qtr_macro["DlyCalDt"].dt.year.astype(str) + "Q" + crsp_qtr_macro["DlyCalDt"].dt.quarter.astype(str)
crsp_qtr_macro.drop(columns = "DlyCalDt", inplace = True)
col = crsp_qtr_macro.pop("YQ")
crsp_qtr_macro.insert(0, "YQ", col)
market_cols = "qtr_ret", "qtr_vola", "qtr_index_level", "qtr_income_ret", "avg_firm_count"
# creating last column market signals to predict current quarter
for col in market_cols: crsp_qtr_macro[f"{col}_lag1"] = crsp_qtr_macro[col].shift(1)
crsp_qtr_macro

C:\Users\sbgka\AppData\Local\Temp\ipykernel_5296\1514321405.py:2: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  crsp_qtr_macro = crsp_daily_macro.resample("Q").agg({


,YQ,qtr_ret,qtr_vola,qtr_index_level,qtr_income_ret,avg_firm_count,qtr_ret_lag1,qtr_vola_lag1,qtr_index_level_lag1,qtr_income_ret_lag1,avg_firm_count_lag1
0,1990Q1,-0.030222,0.008859,604.01,0.008371,499.984127,NaN,NaN,NaN,NaN,NaN
1,1990Q2,0.063453,0.007631,642.34,0.008977,500.000000,-0.030222,0.008859,604.01,0.008371,499.984127
2,1990Q3,-0.139556,0.011900,552.70,0.008717,499.984127,0.063453,0.007631,642.34,0.008977,500.000000
3,1990Q4,0.090365,0.011030,602.64,0.010018,500.000000,-0.139556,0.011900,552.70,0.008717,499.984127
4,1991Q1,0.147108,0.010541,691.30,0.008023,500.000000,0.090365,0.011030,602.64,0.010018,500.000000
...,...,...,...,...,...,...,...,...,...,...,...
119,2019Q4,0.089929,0.005923,10872.55,0.005012,505.000000,0.016904,0.009357,9975.45,0.005074,504.890625
120,2020Q1,-0.194095,0.035693,8762.26,0.005267,504.967742,0.089929,0.005923,10872.55,0.005012,505.000000
121,2020Q2,0.206679,0.019921,10573.22,0.004871,505.000000,-0.194095,0.035693,8762.26,0.005267,504.967742
122,2020Q3,0.089977,0.010664,11524.56,0.004149,505.000000,0.206679,0.019921,10573.22,0.004871,505.000000


When predicting GDP Growth, you are considering the last quarter's market signals

In [117]:
# merging updated macro to train and test sets
train_crsp_macro = train.merge(crsp_qtr_macro, on = "YQ", how = "left")
test_crsp_macro = test.merge(crsp_qtr_macro, on = "YQ", how = "left")

In [118]:
# creating model with cleaned macro data
crsp_macro_model1 = smf.ols("NGDP ~ year + qtr \
                            + qtr_ret + qtr_vola + qtr_index_level + \
                            qtr_income_ret + avg_firm_count \
                            + qtr_ret_lag1 + qtr_vola_lag1 + qtr_index_level_lag1 \
                            + qtr_income_ret_lag1 + avg_firm_count_lag1", train_crsp_macro).fit()
# printing summary of crsp_macro_model1
print(crsp_macro_model1.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.569
Model:                            OLS   Adj. R-squared:                  0.501
Method:                 Least Squares   F-statistic:                     8.304
Date:                Tue, 03 Mar 2026   Prob (F-statistic):           4.87e-11
Time:                        15:36:53   Log-Likelihood:                -204.85
No. Observations:                 103   AIC:                             439.7
Df Residuals:                      88   BIC:                             479.2
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               43.3631 

In [119]:
# get predicted values based off model
y_pred7 = crsp_macro_model1.predict(test_crsp_macro)
y_true7 = test_crsp_macro["NGDP"]
create_submission(test_crsp_macro, y_pred7, "6_stocks_macro1_forecast")

# evaluate using RMSE:
print(f"RMSE Base: {np.sqrt(mean_squared_error(y_true, y_pred))}")
print(f"RMSE w/ Accounting Macro: {np.sqrt(mean_squared_error(y_true2, y_pred2))} (Best on Kaggle)")
print(f"RMSE w/ Stocks Macro: {np.sqrt(mean_squared_error(y_true7, y_pred7))}")

RMSE Base: 2.96495609125275
RMSE w/ Accounting Macro: 4.200668811019781 (Best on Kaggle)
RMSE w/ Stocks Macro: 6.611278292423464


**Stage 1**: removing avg_firm_count (p-value: 0.259), avg_firm_count_lag1 (p-value: 0.347), qtr_ret_lag1 (p-value: 0.751), qtr_income_ret_lag1 (p-value: 0.915)

In [120]:
# creating model with cleaned macro data
crsp_macro_model2 = smf.ols("NGDP ~ year + qtr \
                            + qtr_ret + qtr_vola + qtr_index_level + qtr_income_ret \
                            + qtr_vola_lag1 + qtr_index_level_lag1", train_crsp_macro).fit()
# printing summary of crsp_macro_model2
print(crsp_macro_model2.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.562
Model:                            OLS   Adj. R-squared:                  0.514
Method:                 Least Squares   F-statistic:                     11.80
Date:                Tue, 03 Mar 2026   Prob (F-statistic):           7.98e-13
Time:                        15:36:53   Log-Likelihood:                -205.71
No. Observations:                 103   AIC:                             433.4
Df Residuals:                      92   BIC:                             462.4
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept              158.0987 

In [121]:
# get predicted values based off model
y_pred8 = crsp_macro_model2.predict(test_crsp_macro)
y_true8 = test_crsp_macro["NGDP"]
create_submission(test_crsp_macro, y_pred8, "7_stocks_macro2_forecast")

# evaluate using RMSE:
print(f"RMSE Base: {np.sqrt(mean_squared_error(y_true, y_pred))}")
print(f"RMSE w/ Accounting Macro: {np.sqrt(mean_squared_error(y_true2, y_pred2))} (Best on Kaggle)")
print(f"RMSE w/ Stocks Macro: {np.sqrt(mean_squared_error(y_true7, y_pred7))}")
print(f"RMSE w/ Stocks Macro V2: {np.sqrt(mean_squared_error(y_true8, y_pred8))}")

# model with crsp stocks macro data

RMSE Base: 2.96495609125275
RMSE w/ Accounting Macro: 4.200668811019781 (Best on Kaggle)
RMSE w/ Stocks Macro: 6.611278292423464
RMSE w/ Stocks Macro V2: 5.173724832298544


### Stop point: removing "year" as a parameter
It was recognised that throughout all models, year has a high p-value. It was recognised that year is insignificant as a predictor of GDP due to the test set containing data from 2020. In 2020, due to the COVID-19 pandemic, GDP significantly decreased across countries.

As such, this will not be a predictor of GDP, and a rerun for the base model, accounting macro model and stocks macro model will be done.

In [122]:
# creating model with accounting cleaned macro data
comp_macro_model_f1 = smf.ols("NGDP ~ qtr + atq + niq \
                            + atq_growth + niq_growth \
                            + niq_growth_lag1 + atq_growth_lag1", train_comp_macro).fit()
# printing summary of comp_macro_model_f1
print(comp_macro_model_f1.summary())

# get predicted values based off model
y_pred9 = comp_macro_model_f1.predict(test_comp_macro)
y_true9 = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred9, "8_acct_macro1_forecast")
# evaluate using RMSE:
print()
print(f"RMSE Base: {np.sqrt(mean_squared_error(y_true, y_pred))}")
print(f"RMSE w/ Accounting Macro: {np.sqrt(mean_squared_error(y_true2, y_pred2))} (Best on Kaggle)")
print(f"RMSE w/ Accounting Macro w/o Year: {np.sqrt(mean_squared_error(y_true9, y_pred9))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.391
Model:                            OLS   Adj. R-squared:                  0.331
Method:                 Least Squares   F-statistic:                     6.554
Date:                Tue, 03 Mar 2026   Prob (F-statistic):           3.57e-07
Time:                        15:36:53   Log-Likelihood:                -220.87
No. Observations:                 102   AIC:                             461.7
Df Residuals:                      92   BIC:                             488.0
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           4.3440      0.706     

Insights:
- while it was assumed year is not a significant predictor, it performed worse without it
- RMSE does not determine performance in prediction

### How Analysts' forecast affects GDP (Survey of Professional Forecasters)